# Column Transformer with Mixed Types: scikit-learn vs Moxie

**Reference example:** `../scikit-learn/examples/compose/plot_column_transformer_mixed_types.py` at scikit-learn
`4358542a1bd39af2e7e1cbace8b9fed067473c5b`
([rendered gallery page](https://scikit-learn.org/dev/auto_examples/compose/plot_column_transformer_mixed_types.html)).

**Scenario:** a `ColumnTransformer` routes the numeric Titanic columns (`age`, `fare`) through
`Pipeline(SimpleImputer(strategy="median"), StandardScaler())` and the categorical columns (`embarked`,
`sex`, `pclass`) through `OneHotEncoder` + `SelectPercentile(chi2)`, then `LogisticRegression` classifies.
The example also selects columns by dtype with `make_column_selector`.

**Adaptations:**
- **Data:** `fetch_openml("titanic")` needs the network, so a seeded synthetic Titanic-like frame
  (1309 rows, same five feature columns, NaN in `age` and `fare`, binary `survived`) replaces it. The
  example's Python code then runs unchanged on that frame.
- **Categorical branch is a GAP:** Moxie's `OneHotEncoder` takes typed categorical columns, not an
  `NDArray`, so it cannot be a ColumnTransformer component, and Moxie tables are homogeneous numeric
  matrices, so `make_column_selector(dtype_include="category")` has nothing to select. Both sides therefore
  also run an *adapted* preprocessor where the categorical columns are integer codes passed through.
- **Numeric branch is ADAPTED:** `moxie.pipeline.Pipeline` must end in a predictor, so
  `Pipeline(SimpleImputer, StandardScaler)` cannot be one entry. Moxie chains two ColumnTransformers on
  named tables: (1) `SimpleImputer(median)` on `age`/`fare` + passthrough, `verbose_feature_names_out=False`;
  (2) `StandardScaler` on `age`/`fare` + passthrough, which yields sklearn's exact output and names
  (`num__age`, ..., `cat__pclass`). Each stage is its own `%%mojo` cell (one program per stage keeps
  compilation under the 8 GiB compiler cap); Python only relays the stage-1 array.
- **LogisticRegression is a GAP** in Moxie; the adapted pipeline ends in `GaussianNB` on both sides
  instead, so a full preprocess-then-classify result is still compared.

**Determinism:** deterministic (seeded data, fixed `train_test_split`, closed-form estimators), so
numerical parity is required for the adapted pipeline.


In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "notebooks/_support/moxie_nb.py").exists())
sys.path.insert(0, str(ROOT / "notebooks/_support"))
%load_ext moxie_nb
nb = moxie_nb.Session("compose/plot_column_transformer_mixed_types")

## 1. scikit-learn (Python reference)

Synthetic Titanic stand-in, then the example code.

In [2]:
import numpy as np
import pandas as pd

rng = np.random.RandomState(0)
n = 1309
pclass = rng.choice([1, 2, 3], size=n, p=[0.25, 0.2, 0.55])
sex = rng.choice(["female", "male"], size=n, p=[0.36, 0.64])
embarked = rng.choice(["C", "Q", "S"], size=n, p=[0.2, 0.1, 0.7])
age = np.round(rng.normal(30 - 3 * (pclass - 2), 13, size=n).clip(0.5, 80), 1)
fare = np.round(rng.lognormal(3.6 - 0.6 * (pclass - 1), 0.6, size=n), 2)
age[rng.rand(n) < 0.2] = np.nan
fare[rng.rand(n) < 0.01] = np.nan
logit = 1.2 * (sex == "female") - 0.8 * (pclass - 2) - 0.02 * np.nan_to_num(age - 30) + 0.3 * (embarked == "C") - 0.6
survived = (rng.rand(n) < 1 / (1 + np.exp(-2 * logit))).astype(int)
X = pd.DataFrame({"pclass": pd.Categorical(pclass), "sex": pd.Categorical(sex),
                  "embarked": pd.Categorical(embarked), "age": age, "fare": fare})
y = pd.Series(survived, name="survived")
X.head()

,pclass,sex,embarked,age,fare
0,3,male,S,13.8,18.66
1,3,male,S,27.6,18.32
2,3,female,C,NaN,15.93
3,3,male,S,17.4,9.90
4,2,female,S,54.7,25.02


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectPercentile, chi2
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = ["age", "fare"]
numeric_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)

categorical_features = ["embarked", "sex", "pclass"]
categorical_transformer = Pipeline(
    steps=[
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ("selector", SelectPercentile(chi2, percentile=50)),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)
clf = Pipeline(
    steps=[("preprocessor", preprocessor), ("classifier", LogisticRegression())]
).set_output(transform="pandas")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

clf.fit(X_train, y_train)
print("model score: %.3f" % clf.score(X_test, y_test))
print(list(clf[:-1].get_feature_names_out()))

from sklearn.compose import make_column_selector as selector

subset_feature = ["embarked", "sex", "pclass", "age", "fare"]
X_train, X_test = X_train[subset_feature], X_test[subset_feature]
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, selector(dtype_exclude="category")),
        ("cat", categorical_transformer, selector(dtype_include="category")),
    ]
)
clf = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", LogisticRegression())])
clf.fit(X_train, y_train)
print("model score (dtype selectors): %.3f" % clf.score(X_test, y_test))
print("selected:", selector(dtype_exclude="category")(X_train), selector(dtype_include="category")(X_train))

model score: 0.790
['num__age', 'num__fare', 'cat__sex_female', 'cat__sex_male', 'cat__pclass_1', 'cat__pclass_3']
model score (dtype selectors): 0.790
selected: ['age', 'fare'] ['embarked', 'sex', 'pclass']


Adapted reference that Moxie can express: categorical columns as integer codes passed through,
numeric branch unchanged, `GaussianNB` as the classifier.

In [4]:
from sklearn.naive_bayes import GaussianNB

def codes(frame):
    out = frame[subset_feature].copy()
    for c in categorical_features:
        out[c] = out[c].cat.codes.astype(float)
    return out

C_train, C_test = codes(X_train), codes(X_test)
adapted = ColumnTransformer([("num", numeric_transformer, numeric_features),
                             ("cat", "passthrough", categorical_features)])
sk_train = adapted.fit_transform(C_train)
sk_test = adapted.transform(C_test)
sk_names = adapted.get_feature_names_out()
sk_out = adapted.output_indices_
nbayes = GaussianNB().fit(sk_train, y_train)
sk_proba = nbayes.predict_proba(sk_test)
sk_pred = nbayes.predict(sk_test)
sk_number = selector(dtype_include="number")(C_train)
print(list(sk_names), "| GaussianNB accuracy: %.3f" % nbayes.score(sk_test, y_test))
nb.share(C_train=C_train.to_numpy(), C_test=C_test.to_numpy(), names=np.array(subset_feature),
         y_train=y_train.to_numpy())

['num__age', 'num__fare', 'cat__embarked', 'cat__sex', 'cat__pclass'] | GaussianNB accuracy: 0.771


## 2. Moxie (native Mojo)

Stage 1: median imputation of `age`/`fare` by name on a `ColumnTable`; categorical codes pass through.
`make_column_selector(dtype_include=["number"])` is also resolved against the table.

In [5]:
%%mojo impute
from moxie.compose import ColumnEntry, ColumnTable, ColumnTransformer, make_column_selector
from moxie.impute import SimpleImputer
from moxie_nb_io import load, load_strings, save, save_strings


def main() raises:
    var names = load_strings("names")
    var train = ColumnTable(load("C_train"), names)
    var test = ColumnTable(load("C_test"), names)
    var numeric: List[String] = ["age", "fare"]
    var categorical: List[String] = ["embarked", "sex", "pclass"]
    var stage = ColumnTransformer(
        ColumnEntry("imputer", SimpleImputer(strategy="median"), numeric^),
        ColumnEntry("cat", "passthrough", categorical^),
        verbose_feature_names_out=False,
    )
    save("train", stage.fit_transform(train))
    save("test", stage.transform(test))
    save_strings("names", stage.get_feature_names_out())
    save("statistics", stage.named_transformers_[SimpleImputer[DType.float64]]("imputer").statistics_)
    var key = make_column_selector(dtype_include=List[String](["number"]))(train)
    save_strings("number", key.names.copy())

In [6]:
stage1 = nb.results["impute"]
print("stage-1 names:", list(stage1["names"]), "| medians:", stage1["statistics"])
nb.share(S_train=stage1["train"], S_test=stage1["test"], S_names=stage1["names"])

stage-1 names: [np.str_('age'), np.str_('fare'), np.str_('embarked'), np.str_('sex'), np.str_('pclass')] | medians: [28.95 16.52]


Stage 2: scale the imputed numeric columns by name and classify with `GaussianNB` (typed integer labels).

In [7]:
%%mojo scale_classify
from moxie.compose import ColumnEntry, ColumnTable, ColumnTransformer
from moxie.naive_bayes import GaussianNB
from moxie.preprocessing import StandardScaler
from moxie_nb_io import load, load_ints, load_strings, save, save_ints, save_strings


def main() raises:
    var names = load_strings("S_names")
    var numeric: List[String] = ["age", "fare"]
    var categorical: List[String] = ["embarked", "sex", "pclass"]
    var preprocessor = ColumnTransformer(
        ColumnEntry("num", StandardScaler(), numeric^),
        ColumnEntry("cat", "passthrough", categorical^),
    )
    var train = preprocessor.fit_transform(ColumnTable(load("S_train"), names))
    var test = preprocessor.transform(ColumnTable(load("S_test"), names))
    save("train", train)
    save("test", test)
    save_strings("names", preprocessor.get_feature_names_out())
    var spans = List[Int]()
    for name in ["num", "cat", "remainder"]:
        spans.append(preprocessor.output_indices_.value()[name].start)
        spans.append(preprocessor.output_indices_.value()[name].stop)
    save_ints("spans", spans)
    var model = GaussianNB()
    _ = model.fit(train, load_ints("y_train"))
    save("proba", model.predict_proba(test))
    save_ints("pred", model.predict(test))

Unsupported selection fails clearly: a regex `pattern` selector raises at fit time. (A typed
categorical `OneHotEncoder` component is rejected at compile time, since its input is not an `NDArray`.)

In [8]:
%%mojo pattern_selector --expect-fail
from moxie.compose import ColumnTable, make_column_selector
from moxie_nb_io import load, load_strings


def main() raises:
    var table = ColumnTable(load("C_train"), load_strings("names"))
    _ = make_column_selector(pattern="^(age|fare)$")(table)

Expected failure (unsupported in Moxie):
exit status 1
stack trace was not collected. Enable stack trace collection with environment variable `MODULAR_DEBUG=stack-trace-on-error`
Unhandled exception caught during execution: NotImplementedError: regex column patterns require the shared native regex capability



## 3. Side by side and parity

In [9]:
mx = nb.results["scale_classify"]
moxie_nb.side_by_side(sklearn=pd.DataFrame(sk_test[:5], columns=sk_names),
                      moxie=pd.DataFrame(mx["test"][:5], columns=mx["names"]))
sk_spans = []
for k in ["num", "cat", "remainder"]:
    sk_spans += [sk_out[k].start, sk_out[k].stop]
nb.compare("stage-1 medians (SimpleImputer.statistics_)", adapted.named_transformers_["num"]["imputer"].statistics_,
           stage1["statistics"], exact=True)
nb.compare("make_column_selector(dtype_include='number')", np.array(sk_number), stage1["number"], exact=True)
nb.compare("get_feature_names_out", sk_names, mx["names"], exact=True)
nb.compare("output_indices_ (start, stop) for num/cat/remainder", np.array(sk_spans), mx["spans"], exact=True)
reason_scale = ("StandardScaler: Moxie accumulates moments in Float64 with a different summation order "
                "than NumPy's pairwise mean/var; values are O(1), so differences are a few ulps")
nb.compare("preprocessor fit_transform (train)", sk_train, mx["train"], rtol=1e-12, atol=1e-12, reason=reason_scale)
nb.compare("preprocessor transform (test)", sk_test, mx["test"], rtol=1e-12, atol=1e-12, reason=reason_scale)
nb.compare("GaussianNB predict_proba on preprocessed test", sk_proba, mx["proba"], rtol=1e-9, atol=1e-12,
           reason="same Gaussian likelihoods; variance reductions and the log-sum-exp differ in summation order")
nb.compare("GaussianNB predict", sk_pred, mx["pred"], exact=True)
nb.note("OneHotEncoder + SelectPercentile(chi2) branch", "GAP",
        "Moxie OneHotEncoder takes typed categorical columns, not NDArray, so it cannot be a ColumnTransformer component")
nb.note("make_column_selector(dtype_include='category')", "GAP",
        "native ColumnTable stores one homogeneous numeric matrix; there are no categorical columns to select")
nb.note("Pipeline(SimpleImputer, StandardScaler) as one entry", "ADAPTED",
        "moxie.pipeline.Pipeline must end in a predictor; two chained ColumnTransformers give identical output and names")
nb.note("LogisticRegression", "GAP", "not implemented in moxie.linear_model; GaussianNB is the adapted classifier")
nb.note("set_output(transform='pandas')", "GAP", "pandas output is excluded by the Moxie interface contract")
nb.report()
nb.assert_parity()

,num__age,num__fare,cat__embarked,cat__sex,cat__pclass
0,1.596599,-0.835759,2.0,1.0,2.0
1,-0.107641,-0.451979,2.0,0.0,1.0
2,0.349594,-0.258225,2.0,1.0,2.0
3,-0.440176,0.579664,0.0,1.0,0.0
4,-0.091014,0.551719,2.0,1.0,2.0
,num__age,num__fare,cat__embarked,cat__sex,cat__pclass
0,1.596599,-0.835759,2.0,1.0,2.0
1,-0.107641,-0.451979,2.0,0.0,1.0
2,0.349594,-0.258225,2.0,1.0,2.0
3,-0.440176,0.579664,0.0,1.0,0.0


check,status,sklearn shape,moxie shape,max |diff|,max rel,tolerance,detail,tolerance reason
stage-1 medians (SimpleImputer.statistics_),PASS,"(2,)","(2,)",,,exact,0 mismatches,
make_column_selector(dtype_include='number'),PASS,"(5,)","(5,)",,,exact,0 mismatches,
get_feature_names_out,PASS,"(5,)","(5,)",,,exact,0 mismatches,
"output_indices_ (start, stop) for num/cat/remainder",PASS,"(6,)","(6,)",,,exact,0 mismatches,
preprocessor fit_transform (train),PASS,"(1047, 5)","(1047, 5)",1.29e-14,2.25e-11,"rtol=1e-12, atol=1e-12",,"StandardScaler: Moxie accumulates moments in Float64 with a different summation order than NumPy's pairwise mean/var; values are O(1), so differences are a few ulps"
preprocessor transform (test),PASS,"(262, 5)","(262, 5)",1.24e-14,1.24e-12,"rtol=1e-12, atol=1e-12",,"StandardScaler: Moxie accumulates moments in Float64 with a different summation order than NumPy's pairwise mean/var; values are O(1), so differences are a few ulps"
GaussianNB predict_proba on preprocessed test,PASS,"(262, 2)","(262, 2)",2.55e-15,7.2e-15,"rtol=1e-09, atol=1e-12",,same Gaussian likelihoods; variance reductions and the log-sum-exp differ in summation order
GaussianNB predict,PASS,"(262,)","(262,)",,,exact,0 mismatches,
OneHotEncoder + SelectPercentile(chi2) branch,GAP,,,,,,"Moxie OneHotEncoder takes typed categorical columns, not NDArray, so it cannot be a ColumnTransformer component",
make_column_selector(dtype_include='category'),GAP,,,,,,native ColumnTable stores one homogeneous numeric matrix; there are no categorical columns to select,
